In [ ]:
/**
 * ============================================================
 *  GOOGLE APPS SCRIPT – Jembatan ESP32 ↔ Google Sheets
 * ============================================================
 *  Cara deploy:
 *  1. Buka Google Sheets yang dibuat Colab
 *  2. Klik Extensions → Apps Script
 *  3. Hapus kode default → paste seluruh kode ini
 *  4. Klik Deploy → New deployment
 *  5. Type: Web app
 *     Execute as: Me
 *     Who has access: Anyone
 *  6. Klik Deploy → salin URL yang muncul
 *  7. URL itulah yang diisi ke kode ESP32 #2 (permanen, tidak berubah)
 * ============================================================
 */

// Nama tab tempat URL Colab disimpan
var SHEET_TAB = "Info Server";
// Sel tempat URL Colab ditulis oleh Colab dan dibaca ESP32
var CELL_URL  = "B2";

/**
 * GET request dari ESP32:
 * ESP32 kirim HTTP GET ke URL Apps Script
 * Apps Script baca sel B2 → balas dengan URL Colab
 *
 * Contoh respons:
 * {"status":"ok","url":"https://xxxx.ngrok-free.app/predict"}
 */
function doGet(e) {
  try {
    var ss    = SpreadsheetApp.getActiveSpreadsheet();
    var sheet = ss.getSheetByName(SHEET_TAB);

    if (!sheet) {
      return jsonResponse({status: "error", message: "Tab '" + SHEET_TAB + "' tidak ditemukan"});
    }

    var url = sheet.getRange(CELL_URL).getValue();

    if (!url || url === "" || url === "belum tersedia") {
      return jsonResponse({status: "error", message: "URL belum tersedia. Jalankan Colab dulu!"});
    }

    return jsonResponse({status: "ok", url: url});

  } catch(err) {
    return jsonResponse({status: "error", message: err.toString()});
  }
}

/**
 * POST request dari Colab:
 * Colab kirim URL ngrok baru → Apps Script tulis ke sel B2
 *
 * Body JSON: {"url": "https://xxxx.ngrok-free.app/predict"}
 * Respons  : {"status": "ok", "message": "URL berhasil disimpan"}
 */
function doPost(e) {
  try {
    var data = JSON.parse(e.postData.contents);
    var url  = data.url;

    if (!url) {
      return jsonResponse({status: "error", message: "Parameter 'url' tidak ditemukan"});
    }

    var ss        = SpreadsheetApp.getActiveSpreadsheet();
    var sheet     = ss.getSheetByName(SHEET_TAB);

    if (!sheet) {
      return jsonResponse({status: "error", message: "Tab '" + SHEET_TAB + "' tidak ditemukan"});
    }

    // Tulis URL ke sel B2
    sheet.getRange(CELL_URL).setValue(url);

    // Tulis timestamp update ke sel C2
    sheet.getRange("C2").setValue(new Date().toLocaleString("id-ID"));

    return jsonResponse({status: "ok", message: "URL berhasil disimpan: " + url});

  } catch(err) {
    return jsonResponse({status: "error", message: err.toString()});
  }
}

// Helper: buat respons JSON
function jsonResponse(obj) {
  return ContentService
    .createTextOutput(JSON.stringify(obj))
    .setMimeType(ContentService.MimeType.JSON);
}
